# タイトル：そらふく
氏名：井坂美晴  
使う人：朝、服装に迷っている人  
問い：今日の天気に合った服装は？


In [50]:
# 通信とJSONの読み取りに使う標準ライブラリ
import json
from urllib.request import urlopen
from urllib.parse import urlencode

# ① 調べたい都市名と検索条件を設定する

# 都市名の入力を受け取る（例：Tokyo）
city = input("City: ")

# 取得する候補の数
match_count = 1

# 地名の表示言語
language = "en"

# 通信を待つ最大秒数
timeout = 10

# ② 都市名を検索用のURLに入れ、場所の情報を取得する
url = "https://geocoding-api.open-meteo.com/v1/search?" + urlencode({
    "name": city, "count": match_count, "language": language
})
with urlopen(url, timeout=timeout) as response:
    locations = json.load(response)

# ③ 最初の候補を選び、都市・地域・国が合っているか確認する
place = locations["results"][0]
place["name"], place.get("admin1"), place["country"]

('Tsuchiura', 'Ibaraki', 'Japan')

In [51]:
# ① 前のセルで見つけた場所を使い、取得条件を設定する

# 検索結果の緯度
lat = place["latitude"]

# 検索結果の経度
lon = place["longitude"]

# 過去何日分を含めるか
past_days = 1

# 今日を含めて何日分を取得するか
forecast_days = 3

# その場所の現地時間
timezone = "auto"

# 必要な天気の項目を、カンマでつなぐ
daily_fields = (
    "temperature_2m_max,temperature_2m_min,"
    "precipitation_probability_max,rain_sum,showers_sum"
)

# 通信を待つ最大秒数
timeout = 10

# ② 緯度・経度と取得条件を、天気APIに渡す
url = "https://api.open-meteo.com/v1/forecast?" + urlencode({
    "latitude": lat, "longitude": lon,
    "daily": daily_fields,
    "past_days": past_days, "forecast_days": forecast_days,
    "timezone": timezone
})

# ③ 返ってきたJSONを辞書として読み込む
with urlopen(url, timeout=timeout) as response:
    weather = json.load(response)

# ④ 日別データを表示する
weather["daily"]

{'time': ['2026-11-17', '2026-11-18', '2026-11-19', '2026-11-20'],
 'temperature_2m_max': [13.2, 20.3, 17.9, 14.9],
 'temperature_2m_min': [9.9, 10.4, 8.3, 6.4],
 'precipitation_probability_max': [100, 100, 25, 12],
 'rain_sum': [27.4, 36.3, 0.0, 0.0],
 'showers_sum': [0.0, 0.0, 0.0, 0.0]}

In [52]:
# ① 調べる日と、傘を勧める基準を決める

# 0＝今日、1＝明日
day_offset = 0

# 降水確率の基準（%）
umbrella_threshold = 50

# ② 対象の日の降水確率を取り出す
daily = weather["daily"]
i = past_days + day_offset
chance = daily["precipitation_probability_max"][i]
print(daily["time"][i])

# Noneはデータなし。0%とは区別する
if chance is None:
    print("Rain probability is unavailable.")
else:
    print("Maximum precipitation probability:", chance, "%")

    # ③ 自分たちで決めた基準と比べ、メッセージを変える
    if chance >= umbrella_threshold:
        print("An umbrella may be useful.")
    else:
        print("A lower rain chance, but not a guarantee.")

2026-11-18
Maximum precipitation probability: 100 %
An umbrella may be useful.


In [53]:
# ① 表示を始める日を選ぶ（0＝今日、1＝明日）
start_day = 0

# ② 指定した日から、取得済みの最後の日まで繰り返す
daily = weather["daily"]
for i in range(past_days + start_day, len(daily["time"])):

    # 日付・最低気温・最高気温・降水確率を、同じiで取り出す
    print(daily["time"][i],
          daily["temperature_2m_min"][i], "to",
          daily["temperature_2m_max"][i], "°C |",
          daily["precipitation_probability_max"][i], "%")

2026-11-18 10.4 to 20.3 °C | 100 %
2026-11-19 8.3 to 17.9 °C | 25 %
2026-11-20 6.4 to 14.9 °C | 12 %


In [54]:
# ① 表示する日と、文字の棒の見た目を決める

# 0＝今日から、1＝明日から
start_day = 0

# 棒全体の文字数（正の整数）
bar_width = 20

# 確率に応じて塗る部分
filled = "█"

# 残りの部分
empty = "░"

# ② 100%を棒の長さで割り、1文字分の割合を求める
daily = weather["daily"]
points_per_block = 100 / bar_width
print("Maximum precipitation probability")
print("Each block is about", points_per_block, "percentage points.")

# ③ 各日の降水確率を、文字の棒で表示する
for i in range(past_days + start_day, len(daily["time"])):
    chance = daily["precipitation_probability_max"][i]
    if chance is None:
        print(daily["time"][i], "No data")
    else:

        # 文字数を丸め、塗る部分と残りの部分をつなぐ
        blocks = round(chance / points_per_block)
        bar = filled * blocks + empty * (bar_width - blocks)

        # 丸める前の正確な%も、棒の横に表示する
        print(f'{daily["time"][i]} |{bar}| {chance:3}%')

Maximum precipitation probability
Each block is about 5.0 percentage points.
2026-11-18 |████████████████████| 100%
2026-11-19 |█████░░░░░░░░░░░░░░░|  25%
2026-11-20 |██░░░░░░░░░░░░░░░░░░|  12%


In [55]:
# 今日の気温を取り出す
daily = weather["daily"]
today = past_days

max_temp = daily["temperature_2m_max"][today]
min_temp = daily["temperature_2m_min"][today]

# 気温に合わせてトップスを選ぶ
if max_temp <= 10:
    top = "厚手のニット"
elif max_temp <= 15:
    top = "長袖＋厚手のスウェット"
elif max_temp <= 20:
    top = "長袖シャツ"
elif max_temp <= 25:
    top = "薄手の長袖"
else:
    top = "半袖Tシャツ"

print("おすすめのトップス：", top)

# 朝晩の冷え込みを確認する
if max_temp - min_temp >= 8:
    print("朝晩は冷え込むため、羽織れるものを持っていくと安心です。")
elif min_temp <= 10:
    print("朝晩は寒くなりそうです。暖かい服装を心がけましょう。")
else:
    print("一日を通して比較的過ごしやすそうです。")

おすすめのトップス： 薄手の長袖
朝晩は冷え込むため、羽織れるものを持っていくと安心です。


In [56]:
# 気温に合わせて羽織りものを選ぶ

if min_temp <= 5:
    outer = "厚手のコートやダウンジャケット"
    outer_reason = "朝晩はかなり冷え込むため、防寒できる上着が必要です。"

elif min_temp <= 10:
    outer = "冬用コートや厚手のジャケット"
    outer_reason = "朝晩は冷え込むため、暖かい上着を用意しましょう。"

elif min_temp <= 15:
    outer = "カーディガンやデニムジャケット"
    outer_reason = "朝晩は少し肌寒くなるため、羽織れるものがあると安心です。"

elif max_temp <= 20:
    outer = "薄手のカーディガン"
    outer_reason = "日中も少し肌寒く感じることがありそうです。"

elif max_temp <= 25:
    outer = "薄手のシャツやカーディガン"
    outer_reason = "気温に合わせて脱ぎ着できる服装がおすすめです。"

else:
    outer = "基本的になし"
    outer_reason = "暑くなりそうなので、必要に応じて薄手の羽織りものを用意しましょう。"

print("おすすめの羽織りもの：", outer)
print(outer_reason)

おすすめの羽織りもの： カーディガンやデニムジャケット
朝晩は少し肌寒くなるため、羽織れるものがあると安心です。


In [57]:
# 気温に合わせてボトムスを選ぶ

if max_temp <= 10:
    bottom = "厚手の長ズボン"
    bottom_reason = "足元も冷えやすいため、暖かい素材がおすすめです。"

elif max_temp <= 15:
    bottom = "デニムや厚手の長ズボン"
    bottom_reason = "日中も肌寒くなりそうなので、長ズボンがおすすめです。"

elif max_temp <= 20:
    bottom = "デニムや普通の長ズボン"
    bottom_reason = "一日を通して長ズボンで過ごしやすそうです。"

elif max_temp <= 25:
    bottom = "デニムや薄手のパンツ、スカート"
    bottom_reason = "比較的過ごしやすい気温です。"

else:
    bottom = "薄手のパンツやスカート"
    bottom_reason = "暑くなりそうなので、涼しい素材がおすすめです。"

print("おすすめのボトムス：", bottom)
print(bottom_reason)

おすすめのボトムス： デニムや薄手のパンツ、スカート
比較的過ごしやすい気温です。


In [58]:
# 今日のコーディネートを表示

print("================================")
print("       今日のおすすめコーデ")
print("================================")

print("日付：", daily["time"][today])
print("最高気温：", max_temp, "℃")
print("最低気温：", min_temp, "℃")

print("--------------------------------")
print("👕 トップス：", top)
print("🧥 羽織りもの：", outer)
print("👖 ボトムス：", bottom)
print("--------------------------------")

print("【服装のアドバイス】")
print(outer_reason)
print(bottom_reason)

print("================================")

       今日のおすすめコーデ
日付： 2026-11-18
最高気温： 20.3 ℃
最低気温： 10.4 ℃
--------------------------------
👕 トップス： 薄手の長袖
🧥 羽織りもの： カーディガンやデニムジャケット
👖 ボトムス： デニムや薄手のパンツ、スカート
--------------------------------
【服装のアドバイス】
朝晩は少し肌寒くなるため、羽織れるものがあると安心です。
比較的過ごしやすい気温です。


In [59]:
# 今日の降水確率を取り出す
rain_chance = daily["precipitation_probability_max"][today]

print("【雨のアドバイス】")
print("降水確率：", rain_chance, "%")

if rain_chance is None:
    print("降水確率のデータがありません。")
elif rain_chance >= 70:
    print("雨が降る可能性が高そうです。傘を忘れずに！")
elif rain_chance >= 40:
    print("雨が降るかもしれません。折りたたみ傘があると安心です。")
elif rain_chance >= 20:
    print("雨の可能性は低めですが、念のため天気を確認しましょう。")
else:
    print("雨の可能性は低そうです。")

【雨のアドバイス】
降水確率： 100 %
雨が降る可能性が高そうです。傘を忘れずに！


In [60]:
# 今日の天気とおすすめコーディネートをまとめて表示

print("======================================")
print("          今日なに着る？")
print("======================================")

print("日付：", daily["time"][today])
print("最高気温：", max_temp, "℃")
print("最低気温：", min_temp, "℃")

print("--------------------------------------")
print("【今日のおすすめコーデ】")
print("👕 トップス：", top)
print("🧥 羽織りもの：", outer)
print("👖 ボトムス：", bottom)

print("--------------------------------------")
print("【服装のアドバイス】")
print(outer_reason)
print(bottom_reason)

print("--------------------------------------")
print("【雨のアドバイス】")
print("降水確率：", rain_chance, "%")

if rain_chance is None:
    print("降水確率のデータがないため、天気予報を確認してください。")
elif rain_chance >= 70:
    print("雨が降る可能性が高そうです。傘を持っていきましょう！")
elif rain_chance >= 40:
    print("折りたたみ傘があると安心です。")
elif rain_chance >= 20:
    print("念のため、出かける前に天気を確認しましょう。")
else:
    print("雨の可能性は低そうです。")

print("======================================")

          今日なに着る？
日付： 2026-11-18
最高気温： 20.3 ℃
最低気温： 10.4 ℃
--------------------------------------
【今日のおすすめコーデ】
👕 トップス： 薄手の長袖
🧥 羽織りもの： カーディガンやデニムジャケット
👖 ボトムス： デニムや薄手のパンツ、スカート
--------------------------------------
【服装のアドバイス】
朝晩は少し肌寒くなるため、羽織れるものがあると安心です。
比較的過ごしやすい気温です。
--------------------------------------
【雨のアドバイス】
降水確率： 100 %
雨が降る可能性が高そうです。傘を持っていきましょう！


### 何ができるか
入力した都市の天気予報を取得し、その日の最高気温・最低気温・降水確率をもとに、おすすめの服装を提案するアプリである。トップス、羽織りもの、ボトムスをそれぞれ選び、朝晩の冷え込みや雨の可能性についてもアドバイスを表示する。

### 天気データの出典URL
Open-Meteo
https://api.open-meteo.com/v1/forecast?

### 追加した機能・特徴
気温に合わせてトップスや羽織りもの、ボトムスを選ぶ機能を追加した。また、服装を提案するだけでなく、朝晩の冷え込みに合わせたアドバイスや、降水確率に応じた傘のアドバイスも表示できるようにした点が特徴である。

### これから改善したいこと
今後は風の強さや体感温度なども取り入れ、より実際の天気に合った服装を提案できるようにしたい。また、服装の組み合わせを増やしたり、表示方法を工夫したりして、さらに使いやすいアプリにしていきたいと考えている。